## EEGNet model implementation for 4 classes

In [ ]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())


In [ ]:
import os
import mne
import math
import copy
import gdown
import random
import scipy.io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchsummary import summary
from torch.autograd import Variable
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.utils.data import Dataset, DataLoader
from torchmetrics import Accuracy
from torch.utils.data import Dataset

from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold

In [ ]:
torch.manual_seed(100)
np.random.seed(42)
random.seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

In [ ]:
fs= 200                 #sampling frequency (AFTER CHANGES?)
channel= 32              #electrodes
num_input= 1             #number of channel picture (for EEG signal is always : 1)
num_class= 4             #classes 
signal_length = 267      #number of sample in each AFTER THE DOWNSAMPLING AND SLICING => X = X[:,:,200*2:200*6] and X = X[:,:,0:-1:3]

F1= 8                    #number of temporal filters
D= 3                     #depth multiplier (number of spatial filters)
F2= D*F1                 #number of pointwise filters

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

kernel_size_1= (1,round(fs/2)) 
kernel_size_2= (channel, 1)
kernel_size_3= (1, round(fs/8))
kernel_size_4= (1, 1)

kernel_avgpool_1= (1,4)
kernel_avgpool_2= (1,8)
dropout_rate= 0.2

ks0= int(round((kernel_size_1[0]-1)/2))
ks1= int(round((kernel_size_1[1]-1)/2))
kernel_padding_1= (ks0, ks1-1)
ks0= int(round((kernel_size_3[0]-1)/2))
ks1= int(round((kernel_size_3[1]-1)/2))
kernel_padding_3= (ks0, ks1)

In [ ]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [ ]:
class EEGNet(nn.Module): 
    def __init__(self):
        super().__init__()
        # layer 1
        self.conv2d = nn.Conv2d(num_input, F1, kernel_size_1, padding=kernel_padding_1)
        self.Batch_normalization_1 = nn.BatchNorm2d(F1)
        # layer 2
        self.Depthwise_conv2D = nn.Conv2d(F1, D*F1, kernel_size_2, groups= F1)
        self.Batch_normalization_2 = nn.BatchNorm2d(D*F1)
        self.Elu = nn.ELU()
        self.Average_pooling2D_1 = nn.AvgPool2d(kernel_avgpool_1)
        self.Dropout = nn.Dropout2d(dropout_rate)
        # layer 3
        self.Separable_conv2D_depth = nn.Conv2d( D*F1, D*F1, kernel_size_3,
                                                padding=kernel_padding_3, groups= D*F1)
        self.Separable_conv2D_point = nn.Conv2d(D*F1, F2, kernel_size_4)
        self.Batch_normalization_3 = nn.BatchNorm2d(F2)
        self.Average_pooling2D_2 = nn.AvgPool2d(kernel_avgpool_2)
        # layer 4
        self.Flatten = nn.Flatten()
        self.Dense = nn.Linear(F2*round(signal_length/32), num_class)
        self.Softmax = nn.Softmax(dim= 1)
        
        
    def forward(self, x):
        # layer 1
        y = self.Batch_normalization_1(self.conv2d(x)) #.relu()
        # layer 2
        y = self.Batch_normalization_2(self.Depthwise_conv2D(y))
        y = self.Elu(y)
        y = self.Dropout(self.Average_pooling2D_1(y))
        # layer 3
        y = self.Separable_conv2D_depth(y)
        y = self.Batch_normalization_3(self.Separable_conv2D_point(y))
        y = self.Elu(y)
        y = self.Dropout(self.Average_pooling2D_2(y))
        # layer 4
        y = self.Flatten(y)
        y = self.Dense(y)
        #y = self.Softmax(y) dont need cause using cross entropy loss
        
        return y
    
model = EEGNet()
model.to(device)

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        # X: EEG data (shape: [num_samples, num_channels, signal_length])
        # y: Labels (shape: [num_samples])
        self.X = torch.tensor(X, dtype=torch.float32)  # Convert EEG data to torch tensors
        self.y = torch.tensor(y, dtype=torch.long)     # Convert labels to torch tensors

    def __len__(self):
        # Return the total number of samples
        return len(self.y)

    def __getitem__(self, idx):
        # Return a sample (EEG data and corresponding label)
        return self.X[idx], self.y[idx]


In [ ]:
def train_one_epoch(model, train_loader, loss_fn, optimizer):
    model.train()
    loss_train = AverageMeter()
    acc_train = Accuracy(task="multiclass", num_classes= num_class).to(device)
    
    for i, (inputs, targets) in enumerate(data_loader):
        inputs = inputs.to(device)
        targets = targets.to(device)
        #inputs = inputs.unsqueeze(1)  

        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
        optimizer.zero_grad()

        loss_train.update(loss.item())
        acc_train(outputs, targets.int())
        
    return model, loss_train.avg, acc_train.compute().item()

In [ ]:
file = "features/cleaned_epoched_eeg.npy"
raw_data  =  np.load(file, allow_pickle=True)
print(raw_data.shape)

In [ ]:
unique_classes, class_counts = np.unique(raw_data[:, 0], return_counts=True)
print("Classes:", unique_classes)
print("Counts per class:", class_counts)

In [ ]:
X = np.stack(raw_data[:, 1])
y = raw_data[:, 0].astype(int)
y = y - 1

X = X[:,:,200*2:200*6] 
X = X[:,:,0:-1:3]

print(f"Data shape: {X.shape}, Labels shape: {y.shape}") # better safe then sorry

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

In [ ]:
#normalizing 
train_mean = np.mean(X_train, axis=(0, 2), keepdims=True)  
train_std = np.std(X_train, axis=(0, 2), keepdims=True)    

X_train_normalized = (X_train - train_mean) / train_std
X_val_normalized = (X_val - train_mean) / train_std

print(X_val_normalized.shape)
print(X_train_normalized.shape)

In [ ]:
print("Training set class distribution:", np.bincount(y_train))
print("Validation set class distribution:", np.bincount(y_val))

In [ ]:
import itertools
weight_options = {
    'Left Hand': [1.0, 1.2, 1.5],
    'Right Hand': [1.0, 1.3, 1.6],
    'Feet': [1.0, 1.2, 1.4],
    'Mental Singing': [1.0, 1.2, 1.4]
}
learning_rate_options = [0.001, 0.003, 0.005, 0.01]

# Generate all combinations of class weights and learning rates
combinations = list(itertools.product(weight_options['Left Hand'], 
                                      weight_options['Right Hand'], 
                                      weight_options['Feet'], 
                                      weight_options['Mental Singing'],
                                      learning_rate_options))

best_weights_lr = None
best_val_accuracy = 0.0

In [ ]:
num_folds = 5
kfold = KFold(n_splits=num_folds, shuffle=True, random_state=42)

num_epochs = 50

#class_weights = torch.tensor([1.0, 1.5, 1.0, 1.5], dtype=torch.float32).to(device) 
#loss_fn = nn.CrossEntropyLoss(weight=class_weights)
#loss_fn = nn.CrossEntropyLoss().to(device)

# Iterate through each fold
for i, combination in enumerate(combinations):
    class_weights_values = combination[:4]
    learning_rate = combination[4]
    
    print(f"\nTesting combination {i + 1}/{len(combinations)}: Class Weights={class_weights_values}, Learning Rate={learning_rate}")


    class_weights = torch.tensor(class_weights_values, dtype=torch.float32).to(device)

    train_loss_folds = []
    val_loss_folds = []
    train_acc_folds = []
    val_acc_folds = []

    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train_normalized, y_train)):
    
        print(f'Fold {fold + 1}/{num_folds}')
    
        # Split the data into training and validation sets for this fold
        X_train_fold, y_train_fold = X_train_normalized[train_idx], y_train[train_idx]
        X_val_fold, y_val_fold = X_train_normalized[val_idx], y_train[val_idx]
    
        X_train_fold_reshaped = X_train_fold.reshape(X_train_fold.shape[0], 1, 32, 267)
        X_val_fold_reshaped = X_val_fold.reshape(X_val_fold.shape[0], 1, 32, 267)
    
        X_train_tensor = torch.tensor(X_train_fold_reshaped, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train_fold, dtype=torch.long)
        X_val_tensor = torch.tensor(X_val_fold_reshaped, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val_fold, dtype=torch.long)
    
        # Create DataLoaders for this fold
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    
    
        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    
        # Reinitialize the model and optimizer for each fold
        model = EEGNet().to(device)
        optimizer = optim.NAdam(model.parameters(), lr=learning_rate, weight_decay=1e-3)
        #scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    
        # Reset loss and accuracy history for this fold
        loss_train_hist = []
        acc_train_hist = []
        loss_val_hist = []
        acc_val_hist = []
    
        # Training loop for each fold
        for epoch in range(num_epochs):
            # Training
            model.train()
            train_loss = 0.0
            correct_train = 0
            total_train = 0
    
            for inputs, targets in train_loader:
                inputs, targets = inputs.to(device), targets.to(device)
    
                # Forward pass
                outputs = model(inputs)
                loss = loss_fn(outputs, targets)
    
                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
    
                # Track training loss and accuracy
                train_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_train += targets.size(0)
                correct_train += (predicted == targets).sum().item()
    
            loss_train = train_loss / len(train_loader)
            acc_train = correct_train / total_train
    
            # Validation
            model.eval()
            val_loss = 0.0
            correct_val = 0
            total_val = 0
    
            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs, targets = inputs.to(device), targets.to(device)
    
                    # Forward pass
                    outputs = model(inputs)
                    loss = loss_fn(outputs, targets)
    
                    # Track validation loss and accuracy
                    val_loss += loss.item()
                    _, predicted = torch.max(outputs, 1)
                    total_val += targets.size(0)
                    correct_val += (predicted == targets).sum().item()
    
            loss_val = val_loss / len(val_loader)
            acc_val = correct_val / total_val
    
            loss_train_hist.append(loss_train)
            acc_train_hist.append(acc_train)
            loss_val_hist.append(loss_val)
            acc_val_hist.append(acc_val)
    
            # Update learning rate
            #scheduler.step(loss_val)
    
            if (epoch % 10 == 0) or (epoch % 10 == 5):
                current_lr = optimizer.param_groups[0]['lr']
                print(f'Epoch {epoch}:')
                print(f'  Train Loss: {loss_train:.4f}, Train Accuracy: {acc_train * 100:.2f}%')
                print(f'  Val Loss: {loss_val:.4f}, Val Accuracy: {acc_val * 100:.2f}%')
                print(f'  Current Learning Rate: {current_lr:.6f}\n')
    
        # Store fold performance
        train_loss_folds.append(loss_train_hist[-1])
        val_loss_folds.append(loss_val_hist[-1])
        train_acc_folds.append(acc_train_hist[-1])
        val_acc_folds.append(acc_val_hist[-1])
    
        print(f'Fold {fold + 1} Final Training Accuracy: {acc_train_hist[-1] * 100:.2f}%')
        print(f'Fold {fold + 1} Final Validation Accuracy: {acc_val_hist[-1] * 100:.2f}%')

    avg_train_loss = np.mean(train_loss_folds)
    avg_val_loss = np.mean(val_loss_folds)
    avg_train_acc = np.mean(train_acc_folds)
    avg_val_acc = np.mean(val_acc_folds)

    print(f"Combination {i + 1}/{len(combinations)} - Avg Train Acc: {avg_train_acc * 100:.2f}%, Avg Val Acc: {avg_val_acc * 100:.2f}%")

    # Update the best weights if the current validation accuracy is better
    if avg_val_acc > best_val_accuracy:
        best_val_accuracy = avg_val_acc
        best_weights = class_weights_values

print(f"\nBest class weights: {best_weights} with validation accuracy: {best_val_accuracy * 100:.2f}%")



In [ ]:
# plt.plot(range(num_epochs), acc_train_hist, 'b-', label='Train')  # Train accuracy
# plt.plot(range(num_epochs), acc_val_hist, 'r-', label='Validation')  # Validation accuracy
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy')
# plt.grid(True)
# plt.legend()
# plt.show()

In [ ]:
# all_preds = []
# all_targets = []

# model.eval()
# with torch.no_grad(): 
#     for inputs, targets in val_loader:
#         inputs, targets = inputs.to(device), targets.to(device)
        
#         outputs = model(inputs)
#         _, predicted = torch.max(outputs, 1)
        
#         all_preds.append(predicted.cpu().numpy())
#         all_targets.append(targets.cpu().numpy())


# all_preds = np.concatenate(all_preds)
# all_targets = np.concatenate(all_targets)

# conf_matrix = confusion_matrix(all_targets, all_preds)

# plt.figure(figsize=(10, 7))
# sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Left Hand', 'Right Hand', 'Feet', 'Mental Singing'],
#             yticklabels=['Left Hand', 'Right Hand', 'Feet', 'Mental Singing'])
# plt.xlabel('Predicted Label')
# plt.ylabel('True Label')
# plt.title('Confusion Matrix')
# plt.show()